In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Membuat Fungsi Koreksi di file verify.py**

In [8]:
import os

# Memastikan folder src sudah tersedia
os.makedirs("src", exist_ok=True)

verify_code = """import sqlite3
from datetime import datetime

VALID = {"positif", "negatif", "netral"}

def current_label(conn, comment_id):
    \"\"\"
    Mengambil label terbaru untuk suatu komentar.
    Mengecek apakah sudah pernah dikoreksi di verification_history,
    jika belum, mengambil dari tabel predictions.
    \"\"\"
    cursor = conn.cursor()

    # 1. Cek apakah ada riwayat verifikasi sebelumnya
    cursor.execute(\"\"\"
        SELECT new_label FROM verification_history
        WHERE comment_id = ?
        ORDER BY verified_at DESC LIMIT 1
    \"\"\", (comment_id,))
    row = cursor.fetchone()
    if row:
        return row[0]

    # 2. Jika belum ada koreksi, ambil dari prediksi awal model
    cursor.execute(\"\"\"
        SELECT predicted_label FROM predictions
        WHERE comment_id = ?
        ORDER BY predicted_at DESC LIMIT 1
    \"\"\", (comment_id,))
    row = cursor.fetchone()
    return row[0] if row else "netral"

def add_verification(conn, comment_id, prediction_id, new_label, reason, reviewer_id):
    \"\"\"
    Memvalidasi input koreksi label, mengambil label sebelumnya,
    lalu memasukkannya ke tabel verification_history secara aman (INSERT).
    Tidak ada proses UPDATE pada tabel predictions agar data asli tetap terjaga.
    \"\"\"
    # Validasi 1: Pastikan label baru valid (positif/negatif/netral)
    if new_label not in VALID:
        raise ValueError("Label harus positif, negatif, atau netral")

    # Validasi 2: Pastikan alasan diisi dan minimal 5 karakter
    if not reason or len(reason.strip()) < 5:
        raise ValueError("Alasan minimal 5 karakter")

    # Validasi 3: Pastikan ID pemeriksa wajib diisi
    if not reviewer_id:
        raise ValueError("ID pemeriksa wajib diisi")

    # Ambil label sebelumnya (bisa dari koreksi terakhir atau prediksi awal)
    prev = current_label(conn, comment_id)

    # Masukkan data koreksi baru ke tabel verification_history
    conn.execute(\"\"\"
        INSERT INTO verification_history
        (comment_id, prediction_id, previous_label, new_label, reason, reviewer_id, verified_at)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    \"\"\", (
        comment_id,
        prediction_id,
        prev,
        new_label,
        reason.strip(),
        reviewer_id,
        datetime.now().isoformat(timespec="seconds")
    ))

    conn.commit()
"""

with open("src/verify.py", "w", encoding="utf-8") as f:
    f.write(verify_code)

print("File src/verify.py berhasil dibuat dengan fungsi validasi dan add_verification!")

File src/verify.py berhasil dibuat dengan fungsi validasi dan add_verification!


**Menambahkan Form Koreksi ke file 4_Verifikasi.py**

Membuat tabel verification history terlebih dahulu

In [10]:
import sqlite3

# Hubungkan ke database
conn = sqlite3.connect("data/sentiment.db")
cursor = conn.cursor()

# Buat tabel verification_history jika belum ada
cursor.execute("""
    CREATE TABLE IF NOT EXISTS verification_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        comment_id TEXT,
        prediction_id INTEGER,
        previous_label TEXT,
        new_label TEXT,
        reason TEXT,
        reviewer_id TEXT,
        verified_at TEXT
    )
""")

conn.commit()
conn.close()

print("Tabel verification_history berhasil dibuat di basis data!")

Tabel verification_history berhasil dibuat di basis data!


In [11]:
import os

verifikasi_code = """import sys
from pathlib import Path

# Menambahkan direktori root proyek ke sys.path secara dinamis
root_dir = Path(__file__).resolve().parent.parent.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import streamlit as st
import sqlite3
import pandas as pd
from datetime import datetime
from src.queries import latest_predictions
from src.verify import add_verification, current_label

st.title("🔍 Halaman Verifikasi & Detail Prediksi")
st.write("Gunakan halaman ini untuk memverifikasi detail prediksi, konteks unggahan, dan melakukan koreksi label secara transparan.")

# Koneksi ke database SQLite
def get_connection():
    return sqlite3.connect("data/sentiment.db")

conn = get_connection()

# ==========================================
# ATURAN FORMAT SKOR
# ==========================================
def score_text(score, score_type):
    if score_type in ["probabilitas_softmax", "softmax"]:
        return f"Probabilitas model (belum dikalibrasi): {score:.2f}"
    if score_type == "margin_svm":
        return f"Skor margin SVM: {score:.2f} (bukan probabilitas)"
    if score is not None:
        return f"Skor: {score:.2f} (Tipe: {score_type})"
    return "Skor tidak tersedia"

# Fungsi helper platform
def get_platform_info(comment_id):
    if comment_id.startswith("IG-"):
        return "Instagram"
    elif comment_id.startswith("YT-"):
        return "YouTube"
    elif comment_id.startswith("TT-"):
        return "TikTok"
    return "Media Sosial Lainnya"

# ==========================================
# 1. SIDEBAR FILTER
# ==========================================
st.sidebar.header("Filter Verifikasi")
try:
    versions_df = pd.read_sql("SELECT DISTINCT model_version FROM predictions", conn)
    model_versions = versions_df['model_version'].tolist() if not versions_df.empty else []
except Exception:
    model_versions = []

selected_version = st.sidebar.selectbox("Pilih Versi Model", [None] + model_versions)

df_latest = latest_predictions(conn, selected_version)

if df_latest.empty:
    st.warning("Belum ada data prediksi yang tersimpan di database untuk versi ini.")
else:
    # ==========================================
    # 2. TABEL RINGKAS DI HALAMAN UTAMA
    # ==========================================
    st.subheader("1. Ringkasan Tabel Prediksi Terbaru")
    display_df = df_latest[['comment_id', 'predicted_label', 'score', 'model_version']]
    st.dataframe(display_df, use_container_width=True)

    # ==========================================
    # 3. PANEL DETAIL KOMENTAR & FORM KOREKSI
    # ==========================================
    st.subheader("2. Panel Detail Komentar & Koreksi")
    selected_id = st.selectbox("Pilih Comment ID untuk melihat detail lengkap:", df_latest['comment_id'].tolist())

    if selected_id:
        row = df_latest[df_latest['comment_id'] == selected_id].iloc[0]
        platform_name = get_platform_info(row['comment_id'])

        st.markdown("---")
        col1, col2 = st.columns(2)
        with col1:
            st.markdown(f"**Comment ID:** `{row['comment_id']}`")
            st.markdown(f"**Prediksi Model:** `{row['predicted_label']}`")

            # Tampilkan label aktif saat ini (bisa dari prediksi awal atau riwayat koreksi)
            active_label = current_label(conn, row['comment_id'])
            st.markdown(f"**Label Saat Ini:** `{active_label}`")

            formatted_score = score_text(row['score'], row['score_type'])
            st.markdown(f"**Jenis Skor:** {formatted_score}")
        with col2:
            st.markdown(f"**Versi Model:** `{row['model_version']}`")
            st.markdown(f"**Waktu Prediksi:** `{row['predicted_at']}`")

        st.markdown("**Teks Asli Komentar:**")
        st.info(row['text_original'] if pd.notna(row['text_original']) else "Teks tidak tersedia")

        st.write(f"Platform: **{platform_name}** | Sumber Data: Internal Dataset (`{row['comment_id']}`)")

        # ==========================================
        # FORM KOREKSI (Langkah 2 & 3)
        # ==========================================
        st.markdown("### 📝 Formulir Koreksi Label")

        with st.form("correction_form"):
            col_f1, col_f2 = st.columns(2)
            with col_f1:
                new_label = st.selectbox("Pilih Label Baru:", ["positif", "negatif", "netral"])
            with col_f2:
                reviewer_id = st.selectbox("ID Pemeriksa:", ["reviewer_azza", "reviewer_01", "reviewer_02"])

            reason = st.text_area("Alasan Koreksi (Minimal 5 karakter):", placeholder="Tuliskan alasan mengapa label ini dikoreksi...")

            submitted = st.form_submit_button("Simpan Koreksi")

            if submitted:
                try:
                    # Memanggil fungsi add_verification yang ada di src/verify.py
                    prediction_id = int(row['id']) if 'id' in row and pd.notna(row['id']) else 1
                    add_verification(
                        conn=conn,
                        comment_id=row['comment_id'],
                        prediction_id=prediction_id,
                        new_label=new_label,
                        reason=reason,
                        reviewer_id=reviewer_id
                    )
                    timestamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    st.success(f"Koreksi berhasil disimpan pada {timestamp_now}! Label baru: **{new_label}** (Pemeriksa: {reviewer_id})")
                except ValueError as ve:
                    st.error(f"Validasi Gagal: {ve}")
                except Exception as e:
                    st.error(f"Terjadi kesalahan pada basis data: {e}")

conn.close()
"""

with open("app/pages/4_Verifikasi.py", "w", encoding="utf-8") as f:
    f.write(verifikasi_code)

print("File app/pages/4_Verifikasi.py berhasil diperbarui dengan Form Koreksi!")

File app/pages/4_Verifikasi.py berhasil diperbarui dengan Form Koreksi!


**Pengujian 10 koreksi komentar**

In [15]:
import pandas as pd
import os

# Memastikan direktori docs tersedia
os.makedirs("docs", exist_ok=True)

print("=== PENGUJIAN LANGKAH 4: PENGUJIAN 10 KOREKSI KOMENTAR ===\n")

# Menyiapkan 10 data koreksi beserta alasan validnya
data = [
    {
        "comment_id": "IG-P004-C001",
        "prediksi": "negatif",
        "label koreksi": "positif",
        "alasan": "Ekspresi dukungan positif terhadap program"
    },
    {
        "comment_id": "IG-P004-C002",
        "prediksi": "positif",
        "label koreksi": "netral",
        "alasan": "Komentar hanya berupa emoji tanpa teks pendukung, sehingga nuansa sentimennya lebih tepat dikategorikan netral."
    },
    {
        "comment_id": "IG-P004-C003",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Penggunaan kata sifat 'Indah' disertai emoji api (🔥🔥🔥) menunjukkan kekaguman dan apresiasi yang kuat terhadap suatu tempat."
    },
    {
        "comment_id": "IG-P004-C004",
        "prediksi": "positif",
        "label koreksi": "netral",
        "alasan": "Observasi situasi bersifat jenaka dan santai."
    },
    {
        "comment_id": "IG-P004-C005",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ungkapan rasa syukur dan kebanggaan atas prestasi."
    },
    {
        "comment_id": "IG-P004-C006",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ekspresi syukur dan kekaguman bernada positif."
    },
    {
        "comment_id": "IG-P004-C007",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ungkapan rasa bangga dan pujian bernada positif."
    },
    {
        "comment_id": "IG-P004-C008",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ekspresi rasa syukur dan kecintaan bernada positif."
    },
    {
        "comment_id": "IG-P004-C009",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ungkapan rasa syukur yang sangat mendalam."
    },
    {
        "comment_id": "IG-P004-C010",
        "prediksi": "netral",
        "label koreksi": "positif",
        "alasan": "Ekspresi antusiasme dan apresiasi kesesuaian konsep."
    }
]

# Mengubah data ke dalam DataFrame pandas
df = pd.DataFrame(data)

# Menyimpan ke dalam file CSV di folder docs
df.to_csv("docs/D22_koreksi_contoh.csv", index=False)

print("File docs/D22_koreksi_contoh.csv berhasil dibuat dengan 10 data koreksi!")
print(df)

=== PENGUJIAN LANGKAH 4: PENGUJIAN 10 KOREKSI KOMENTAR ===

File docs/D22_koreksi_contoh.csv berhasil dibuat dengan 10 data koreksi!
     comment_id prediksi label koreksi  \
0  IG-P004-C001  negatif       positif   
1  IG-P004-C002  positif        netral   
2  IG-P004-C003   netral       positif   
3  IG-P004-C004  positif        netral   
4  IG-P004-C005   netral       positif   
5  IG-P004-C006   netral       positif   
6  IG-P004-C007   netral       positif   
7  IG-P004-C008   netral       positif   
8  IG-P004-C009   netral       positif   
9  IG-P004-C010   netral       positif   

                                              alasan  
0         Ekspresi dukungan positif terhadap program  
1  Komentar hanya berupa emoji tanpa teks penduku...  
2  Penggunaan kata sifat 'Indah' disertai emoji a...  
3      Observasi situasi bersifat jenaka dan santai.  
4  Ungkapan rasa syukur dan kebanggaan atas prest...  
5     Ekspresi syukur dan kekaguman bernada positif.  
6   Ungkapan rasa b

**Pengujian validasi input tidak valid**

In [18]:
import sqlite3
from src.verify import add_verification

# Hubungkan ke database SQLite
conn = sqlite3.connect("data/sentiment.db")

print("=== PENGUJIAN LANGKAH 5: UJI VALIDASI INPUT TIDAK VALID ===\n")

# 1. Uji Label Tidak Valid (di luar pilihan "positif", "negatif", "netral")
try:
    print("Menguji label salah ('sangat_bagus')...")
    add_verification(conn, "IG-P004-C001", 1, "sangat_bagus", "Alasan valid lebih dari 5 karakter", "reviewer_azza")
except ValueError as e:
    print("BERHASIL MENANGKAP ERROR LABEL:", e)

print("-" * 50)

# 2. Uji Alasan Kosong ("")
try:
    print("Menguji alasan kosong ('')...")
    add_verification(conn, "IG-P004-C001", 1, "positif", "", "reviewer_azza")
except ValueError as e:
    print("BERHASIL MENANGKAP ERROR ALASAN KOSONG:", e)

print("-" * 50)

# 3. Uji Alasan Terlalu Pendek / Kurang dari 5 Karakter
try:
    print("Menguji alasan terlalu pendek ('oke')...")
    add_verification(conn, "IG-P004-C001", 1, "positif", "oke", "reviewer_azza")
except ValueError as e:
    print("BERHASIL MENANGKAP ERROR ALASAN:", e)

conn.close()

=== PENGUJIAN LANGKAH 5: UJI VALIDASI INPUT TIDAK VALID ===

Menguji label salah ('sangat_bagus')...
BERHASIL MENANGKAP ERROR LABEL: Label harus positif, negatif, atau netral
--------------------------------------------------
Menguji alasan kosong ('')...
BERHASIL MENANGKAP ERROR ALASAN KOSONG: Alasan minimal 5 karakter
--------------------------------------------------
Menguji alasan terlalu pendek ('oke')...
BERHASIL MENANGKAP ERROR ALASAN: Alasan minimal 5 karakter
